# Decoupled Delta-Net: Google Colab Runner
**Project:** Sub-quadratic linear recurrence with surprise-triggered error reservoir
**Author:** Jaime Jarauta
**Pre-Registered Question:** Can decoupled delta recurrence maintain associative recall at > 150 tok/s on commodity GPUs/CPUs?

In [ ]:
# Step 1: Hardware Pre-flight & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'decoupled-delta-net'
    if not os.path.exists(repo_dir):
        print('Cloning decoupled-delta-net (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/decoupled-delta-net.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication
# Securely reads from Colab Secrets (Key icon on left sidebar) or reuses ~/.netrc.
import os
import wandb

# 1. Attempt reading from Google Colab Secrets across candidate names
try:
    from google.colab import userdata
    for secret_name in ['WANDB_API_KEY', 'wandb_api_key', 'WANDB', 'WAN', 'wandb']:
        try:
            val = userdata.get(secret_name)
            if val and len(val.strip()) > 10:
                os.environ['WANDB_API_KEY'] = val.strip()
                print(f"[W&B] Loaded API key from Colab Secrets ('{secret_name}').")
                break
        except Exception:
            pass
except Exception:
    pass

# 2. Fallback: check ~/.netrc if previously logged in
netrc_path = os.path.expanduser('~/.netrc')
if 'WANDB_API_KEY' not in os.environ and os.path.exists(netrc_path):
    try:
        with open(netrc_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2 and parts[0] == 'password' and len(parts[1]) > 10:
                    os.environ['WANDB_API_KEY'] = parts[1]
                    print("[W&B] Restored WANDB_API_KEY automatically from ~/.netrc.")
                    break
    except Exception:
        pass

# 3. Login to W&B without prompting if credentials exist
if 'WANDB_API_KEY' in os.environ:
    wandb.login(key=os.environ['WANDB_API_KEY'], relogin=False)
else:
    print("[W&B] Note: To avoid interactive prompt, name your secret 'WANDB_API_KEY' and paste full key into Value.")
    wandb.login(relogin=False)

os.environ['WANDB_ENTITY'] = 'research-projekts'
print('[W&B] Authenticated successfully. Target entity: research-projekts')


In [ ]:
# Step 2: Dependencies & Ingestion
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("decoupled-delta-net"):
        os.system("git clone -q -b feature/initial-implementation https://github.com/jaimejarauta/decoupled-delta-net.git")
    if "decoupled-delta-net/src" not in sys.path:
        sys.path.insert(0, "decoupled-delta-net/src")

if "src" not in sys.path:
    sys.path.insert(0, "src")

try:
    from decoupled_delta.model import DecoupledDeltaModel
    from decoupled_delta.metrics import benchmark_associative_throughput
    print("[Ready] Decoupled Delta-Net loaded successfully from package.")
except ImportError:
    print("[Fallback] Loading standalone Decoupled Delta-Net engine...")

"""Decoupled Delta-Net Architecture: Linear recurrence with surprise-triggered error reservoir."""

import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class ErrorReservoir:
    """Fixed-capacity circular reservoir caching high-surprise prediction residuals."""

    def __init__(self, capacity: int = 128, d_k: int = 32, d_v: int = 32, device: torch.device = torch.device("cpu"), dtype: torch.dtype = torch.float32):
        self.capacity = capacity
        self.d_k = d_k
        self.d_v = d_v
        self.device = device
        self.dtype = dtype
        self.k_buffer = torch.zeros(capacity, d_k, device=device, dtype=dtype)
        self.e_buffer = torch.zeros(capacity, d_v, device=device, dtype=dtype)
        self.ptr = 0
        self.count = 0

    def push(self, k: torch.Tensor, e: torch.Tensor):
        """Append a surprise key-error pair to the circular reservoir."""
        # k: (d_k,), e: (d_v,)
        self.k_buffer[self.ptr] = k.detach().to(self.dtype)
        self.e_buffer[self.ptr] = e.detach().to(self.dtype)
        self.ptr = (self.ptr + 1) % self.capacity
        self.count = min(self.count + 1, self.capacity)

    def query(self, q: torch.Tensor) -> torch.Tensor:
        """Associative readout from reservoir: q in R^{d_k} -> readout in R^{d_v}."""
        if self.count == 0:
            return torch.zeros(self.d_v, device=self.device, dtype=self.dtype)
        valid_k = self.k_buffer[:self.count]  # (C, d_k)
        valid_e = self.e_buffer[:self.count]  # (C, d_v)
        scores = torch.matmul(valid_k, q.to(self.dtype)) / math.sqrt(self.d_k)  # (C,)
        weights = F.softmax(scores, dim=0)  # (C,)
        return torch.matmul(weights, valid_e)  # (d_v,)


class DecoupledDeltaCell(nn.Module):
    """Linear delta-rule recurrent cell coupled with a discrete surprise reservoir."""

    def __init__(
        self,
        d_model: int = 64,
        d_k: int = 32,
        d_v: int = 32,
        surprise_threshold: float = 0.5,
        reservoir_capacity: int = 64,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.surprise_threshold = surprise_threshold
        self.reservoir_capacity = reservoir_capacity

        self.proj_q = nn.Linear(d_model, d_k, bias=False)
        self.proj_k = nn.Linear(d_model, d_k, bias=False)
        self.proj_v = nn.Linear(d_model, d_v, bias=False)
        self.proj_beta = nn.Linear(d_model, 1, bias=True)

        self.out_proj = nn.Linear(d_v, d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """Processes sequence (B, T, d_model) through decoupled delta recurrence.
        
        Accelerated with in-place fused state updates and batched reservoir processing,
        eliminating sequential batch loops and redundant inner tensor allocations.
        """
        b, t, d = x.shape
        device = x.device
        dtype = x.dtype

        q = self.proj_q(x)                            # (B, T, d_k)
        k = F.normalize(self.proj_k(x), p=2, dim=-1)  # (B, T, d_k)
        v = self.proj_v(x)                            # (B, T, d_v)
        beta = torch.sigmoid(self.proj_beta(x))        # (B, T, 1)

        y_out = torch.empty(b, t, self.d_v, device=device, dtype=dtype)
        surprises_logged = 0
        thresh_sq = self.surprise_threshold ** 2
        scale = 1.0 / math.sqrt(self.d_k)

        if b == 1:
            s_state = torch.zeros(self.d_k, self.d_v, device=device, dtype=dtype)
            reservoir = ErrorReservoir(capacity=self.reservoir_capacity, d_k=self.d_k, d_v=self.d_v, device=device, dtype=dtype)
            k_buf = reservoir.k_buffer
            e_buf = reservoir.e_buffer
            q_0 = q[0]
            k_0 = k[0]
            v_0 = v[0]
            beta_0 = beta[0]
            ptr = 0
            count = 0
            cap = self.reservoir_capacity

            for step in range(t):
                q_t = q_0[step]
                k_t = k_0[step]
                v_t = v_0[step]
                b_t = beta_0[step]

                v_pred = torch.matmul(k_t, s_state)
                err = v_t - v_pred
                s_state.addmm_(k_t.unsqueeze(1), (b_t * err).unsqueeze(0))
                y_fast = torch.matmul(q_t, s_state)

                err_sq = torch.dot(err, err).item()
                if err_sq > thresh_sq:
                    k_buf[ptr] = k_t
                    e_buf[ptr] = err
                    ptr = (ptr + 1) % cap
                    count = min(count + 1, cap)
                    surprises_logged += 1

                if count == 0:
                    y_out[0, step] = y_fast
                else:
                    scores = torch.matmul(k_buf[:count], q_t) * scale
                    weights = F.softmax(scores, dim=0)
                    y_slow = torch.matmul(weights, e_buf[:count])
                    y_out[0, step] = y_fast + y_slow
        else:
            s_state = torch.zeros(b, self.d_k, self.d_v, device=device, dtype=dtype)
            k_buf = torch.zeros(b, self.reservoir_capacity, self.d_k, device=device, dtype=dtype)
            e_buf = torch.zeros(b, self.reservoir_capacity, self.d_v, device=device, dtype=dtype)
            valid_mask = torch.zeros(b, self.reservoir_capacity, dtype=torch.bool, device=device)
            ptr = torch.zeros(b, dtype=torch.long, device=device)
            batch_indices = torch.arange(b, device=device)
            cap = self.reservoir_capacity

            for step in range(t):
                q_t = q[:, step]
                k_t = k[:, step]
                v_t = v[:, step]
                b_t = beta[:, step]

                v_pred = torch.bmm(k_t.unsqueeze(1), s_state).squeeze(1)
                err = v_t - v_pred
                s_state.baddbmm_(k_t.unsqueeze(2), (b_t * err).unsqueeze(1))
                y_fast = torch.bmm(q_t.unsqueeze(1), s_state).squeeze(1)

                err_sq = torch.sum(err * err, dim=-1)
                is_surprise = err_sq > thresh_sq
                surprises_logged += int(is_surprise.sum().item())

                curr_ptr = ptr
                k_buf[batch_indices, curr_ptr] = torch.where(is_surprise.unsqueeze(-1), k_t, k_buf[batch_indices, curr_ptr])
                e_buf[batch_indices, curr_ptr] = torch.where(is_surprise.unsqueeze(-1), err, e_buf[batch_indices, curr_ptr])
                valid_mask[batch_indices, curr_ptr] = is_surprise | valid_mask[batch_indices, curr_ptr]
                ptr = (curr_ptr + is_surprise.long()) % cap

                scores = torch.bmm(k_buf, q_t.unsqueeze(-1)).squeeze(-1) * scale
                scores = scores.masked_fill(~valid_mask, -1e4)
                weights = F.softmax(scores, dim=-1)
                y_slow = torch.bmm(weights.unsqueeze(1), e_buf).squeeze(1)
                has_valid = valid_mask.any(dim=-1, keepdim=True)
                y_slow = torch.where(has_valid, y_slow, torch.zeros_like(y_slow))
                y_out[:, step] = y_fast + y_slow

        out = self.out_proj(y_out)
        telemetry = {
            "surprise_rate": surprises_logged / (b * t + 1e-7),
        }
        return out, telemetry


class DecoupledDeltaBlock(nn.Module):
    """Full Decoupled Delta-Net layer with pre-norm, cell, and MLP."""

    def __init__(
        self,
        d_model: int = 64,
        d_k: int = 32,
        d_v: int = 32,
        d_ff: int = 128,
        surprise_threshold: float = 0.5,
        dropout: float = 0.05,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.cell = DecoupledDeltaCell(
            d_model=d_model, d_k=d_k, d_v=d_v, surprise_threshold=surprise_threshold
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        h, telem = self.cell(self.norm1(x))
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x, telem


class DecoupledDeltaModel(nn.Module):
    """Decoupled Delta-Net language and sequence model."""

    def __init__(
        self,
        vocab_size: int = 256,
        d_model: int = 64,
        d_k: int = 32,
        d_v: int = 32,
        d_ff: int = 128,
        num_layers: int = 2,
        surprise_threshold: float = 0.5,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            DecoupledDeltaBlock(
                d_model=d_model, d_k=d_k, d_v=d_v, d_ff=d_ff, surprise_threshold=surprise_threshold
            )
            for _ in range(num_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        x = self.embed(input_ids)
        total_telem = {}
        for idx, layer in enumerate(self.layers):
            x, telem = layer(x)
            total_telem[f"layer_{idx}_surprise_rate"] = telem["surprise_rate"]
        logits = self.head(x)
        return logits, total_telem

"""Metrics and evaluation benchmarks for Decoupled Delta-Net."""

import time
from typing import Dict
import torch


def compute_mqar_accuracy(logits: torch.Tensor, target_val: torch.Tensor) -> float:
    """Computes associative recall top-1 accuracy at query position."""
    # logits: (B, T, vocab_size), target_val: (B,)
    preds = torch.argmax(logits[:, -1, :], dim=-1)
    correct = (preds == target_val).float().sum().item()
    return float(correct / len(target_val))


def benchmark_associative_throughput(
    model: torch.nn.Module,
    seq_len: int = 2048,
    batch_size: int = 2,
    device: torch.device = torch.device("cpu"),
) -> Dict[str, float]:
    """Measures streaming inference throughput and surprise statistics."""
    model.eval()
    x = torch.randint(10, 200, (batch_size, seq_len), device=device)

    # Warmup
    with torch.no_grad():
        _ = model(x[:, :64])

    t0 = time.perf_counter()
    with torch.no_grad():
        logits, telem = model(x)
    elapsed = time.perf_counter() - t0

    tokens = seq_len * batch_size
    tokens_per_sec = tokens / elapsed
    ms_per_token = (elapsed / tokens) * 1000.0

    return {
        "tokens_per_sec": tokens_per_sec,
        "ms_per_token": ms_per_token,
        "elapsed_sec": elapsed,
        "seq_len": seq_len,
        "surprise_rate": telem.get("layer_0_surprise_rate", 0.0),
    }



In [ ]:
# Step 3: Run Smoke Benchmark
model = DecoupledDeltaModel(vocab_size=256, d_model=64, d_k=32, d_v=32, num_layers=2).to(device)
metrics = benchmark_associative_throughput(model, seq_len=1024, batch_size=2, device=device)
print("=" * 60)
print(f"Throughput : {metrics['tokens_per_sec']:.1f} tok/s (Target: >= 150 tok/s)")
print(f"Per-Token Latency : {metrics['ms_per_token']:.3f} ms/token")
print(f"Surprise Rate : {metrics['surprise_rate'] * 100:.1f}%")
gate_passed = metrics['tokens_per_sec'] >= 150 and metrics['ms_per_token'] <= 10.0
print(f"Gate 1 Status: {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 3b: Long-Horizon Associative Retrieval & Surprise Reservoir Benchmark
!pytest tests
!python scripts/run_smoke.py


In [ ]:
# Step Pre-Kill: Explicit W&B Telemetry Cloud Upload
# Ensures every run (online or offline) is completely synced and uploaded to W&B cloud before teardown
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true
!wandb sync --include-offline /tmp/wandb wandb 2>/dev/null || true


In [ ]:
# Step Final: Credit Protection & Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
# Set to False by default so the virtual machine stays alive after execution
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
else:
    print("\n[Credit Guard] AUTO_KILL_ON_FINISH is False. Keeping VM instance alive for inspection.")
